In [1]:
import sys
from pathlib import Path
# Ensure ml-model package is importable when running from repository root
sys.path.append(str(Path.cwd() / "ml-model"))
import json
from models import (
    AttackCategory,
    CyberAttackEvolutionModel,
    EvolutionEvent,
    PredictionResult,
    TimeSeriesPoint,
)

__all__ = [
    "AttackCategory",
    "TimeSeriesPoint",
    "PredictionResult",
    "EvolutionEvent",
    "CyberAttackEvolutionModel",
]

In [2]:
# Optional: set `pickle_path` to load a previously saved model (.pkl).
# Example: pickle_path = 'trained_models/cyber_model_20240620_123456.pkl'
pickle_path = None  # set to a string path to load
model = None
if pickle_path:
    try:
        model = CyberAttackEvolutionModel.load_from_pickle(pickle_path)
        print(f'Loaded model from {pickle_path}')
    except Exception as e:
        print('Failed to load pickle:', e)

## 1) Initialize model

Create a `CyberAttackEvolutionModel` instance. If the historical data file is missing or invalid the constructor will raise — fix the data file shown in `ml-model/historical_attack_data.template.json` then re-run this cell.

In [3]:
# Initialize model (adjust `lookback_months` as needed)
try:
    model = CyberAttackEvolutionModel(lookback_months=24)
    print('✓ Model initialized')
    print(f'Loaded categories: {len(model.historical_data)}')
except (FileNotFoundError, ValueError) as error:
    model = None
    print(f'✗ Initialization failed: {error}')
    print('Provide historical data in ml-model/historical_attack_data.json or use the template file.')

✓ Model initialized
Loaded categories: 10


## 2) Train the model

Train the model and inspect basic metrics. Training is implemented in the model class and returns a metrics dictionary (e.g., model type, R², MAE, RMSE).

In [4]:
if model is None:
    print('Model not initialized — run the initialization cell first.')
else:
    print('Training model...')
    metrics = model.train()
    print('✓ Model trained')
    print(metrics)

Training model...
✓ Model trained
{'model_type': 'Deterministic Time-Series Baseline', 'training_samples': 120, 'categories_trained': 10, 'lookback_window': '24 months', 'training_loss': 0.0166, 'validation_loss': 0.0182, 'mae': 1.62, 'rmse': 1.77, 'r2_score': 0.9991, 'convergence_epochs': 120}


## 3) Generate Predictions

Generate predictions for a horizon (months). The notebook prints a simple table — you can extend this cell to use `pandas` or visualization libraries for richer output.

In [6]:
if model is None:
    print('Model not initialized — run the initialization cell first.')
else:
    horizon = 6
    print(f'Generating {horizon}-month predictions...')
    predictions = model.predict(horizon_months=horizon)
    # Print header
    print(f"{'Category':<20} {'Current':>10} {'Predicted':>10} {'Change':>8} {'Risk':>10} {'Conf':>6}")
    print('-' * 70)
    for p in predictions:
        cat = getattr(p, 'category', p.get('category') if isinstance(p, dict) else str(p))
        current = getattr(p, 'current_value', p.get('current_value', 0) if isinstance(p, dict) else 0)
        predicted = getattr(p, 'predicted_value', p.get('predicted_value', 0) if isinstance(p, dict) else 0)
        change = getattr(p, 'change_pct', p.get('change_pct', 0) if isinstance(p, dict) else 0)
        risk = getattr(p, 'risk_level', p.get('risk_level', 'unknown') if isinstance(p, dict) else 'unknown')
        conf = getattr(p, 'confidence', p.get('confidence', 0) if isinstance(p, dict) else 0)
        print(f"{cat:<20} {current:>10.1f} {predicted:>10.1f} {change:>+7.1f}% {risk:>10} {conf:>5.1%}")

Generating 6-month predictions...
Category                Current  Predicted   Change       Risk   Conf
----------------------------------------------------------------------
phishing                  229.7      232.1    +1.1%       high 98.0%
ai_powered                 55.7       62.4   +12.1%     medium 95.9%
iot_exploit                59.0       62.7    +6.3%        low 97.5%
supply_chain               69.0       72.6    +5.3%        low 97.9%
zero_day                   35.5       37.3    +5.1%        low 97.9%
ransomware                137.0      140.7    +2.7%        low 98.0%
insider_threat             71.5       73.1    +2.3%        low 98.0%
apt                        98.7      100.8    +2.2%        low 98.0%
ddos                      176.5      178.1    +0.9%        low 98.0%
cryptojacking              76.5       75.1    -1.8%        low 98.0%


## 4) Detect Evolution Events

Run the model's event detection to see parent→child pattern mutations.

In [9]:
if model is None:
    print('Model not initialized — run the initialization cell first.')
else:
    events = model.detect_evolution_events()
    if not events:
        print('No evolution events detected.')
    else:
        print('Detected Evolution Events:')
        print('-' * 70)
        for e in events:
            # Support both dataclass/object and dict representations without calling .get on objects
            if isinstance(e, dict):
                ts = e.get('timestamp', '')
                parent = e.get('parent_pattern', '')
                child = e.get('child_pattern', '')
                mtype = e.get('mutation_type', '')
                conf = e.get('confidence', 0)
            else:
                ts = getattr(e, 'timestamp', '')
                parent = getattr(e, 'parent_pattern', '')
                child = getattr(e, 'child_pattern', '')
                mtype = getattr(e, 'mutation_type', '')
                conf = getattr(e, 'confidence', 0)
            print(f'  [{ts}] {parent} → {child}')
            print(f'    Type: {mtype} | Confidence: {conf:.0%}')
            print()

Detected Evolution Events:
----------------------------------------------------------------------
  [2025-12-01] supply_chain + zero_day → Combined Supply Chain-Zero Day
    Type: convergence | Confidence: 85%

  [2025-12-01] supply_chain + iot_exploit → Combined Supply Chain-Iot Exploit
    Type: convergence | Confidence: 85%

  [2025-12-01] supply_chain + ai_powered → Combined Supply Chain-Ai Powered
    Type: convergence | Confidence: 85%

  [2025-12-01] zero_day + iot_exploit → Combined Zero Day-Iot Exploit
    Type: convergence | Confidence: 85%

  [2025-12-01] zero_day + ai_powered → Combined Zero Day-Ai Powered
    Type: convergence | Confidence: 85%

  [2025-12-01] iot_exploit + ai_powered → Combined Iot Exploit-Ai Powered
    Type: convergence | Confidence: 85%

